In [1]:
# --- Imports for psycopg2 approach ---
import pandas as pd  # read SQL results into DataFrames
import psycopg2  # PostgreSQL adapter
import os
from dotenv import load_dotenv  # read credentials from .env file

print("Packages loaded!")

Packages loaded!


In [3]:
# --- Load credentials from .env ---
# load_dotenv() reads the .env file and makes variables available via os.getenv()
load_dotenv()

DATABASE = os.getenv("DATABASE")
USER_DB = os.getenv("USER_DB")
PASSWORD = os.getenv("PASSWORD")
HOST = os.getenv("HOST")
PORT = os.getenv("PORT")

# Quick check (never print actual credentials!)
print(f"Database: {DATABASE}")
print(f"Host:     {HOST}")
print(f"Port:     {PORT}")

Database: postgres
Host:     ds-sql-playground.c8g8r1deus2v.eu-central-1.rds.amazonaws.com
Port:     5432


In [10]:
# --- Open connection to the database ---
conn = psycopg2.connect(
    database=DATABASE, user=USER_DB, password=PASSWORD, host=HOST, port=PORT
)

print("Connection established!")

Connection established!


In [11]:
# --- Create a cursor and run a test query ---
cur = conn.cursor()  # create cursor

cur.execute("""select p.*, bm.insulin, bm.glucose , bm.bloodpressure,
po.diabetespedigreefunction , po.outcome,
sk.skinthickness 
from diabetes.patient p
inner join diabetes.blood_metrics bm on bm.patientid = p.id
inner join diabetes.pedigree_outcome po on po.patientid = p.id
inner join diabetes.skin sk on sk.patientid = p.id; 
""")  # send SQL
rows = cur.fetchall()  # retrieve results

print(f"Retrieved {len(rows)} rows")
for row in rows:
    print(row)

Retrieved 1536 rows
(1, 50, 6, 33.6, 0, 148, 72, 0.627, 1, 35)
(2, 31, 1, 26.6, 0, 85, 66, 0.351, 0, 29)
(3, 32, 8, 23.3, 0, 183, 64, 0.672, 1, 0)
(4, 21, 1, 28.1, 94, 89, 66, 0.167, 0, 23)
(5, 33, 0, 43.1, 168, 137, 40, 2.288, 1, 35)
(6, 30, 5, 25.6, 0, 116, 74, 0.201, 0, 0)
(7, 26, 3, 31.0, 88, 78, 50, 0.248, 1, 32)
(8, 29, 10, 35.3, 0, 115, 0, 0.134, 0, 0)
(9, 53, 2, 30.5, 543, 197, 70, 0.158, 1, 45)
(10, 54, 8, 0.0, 0, 125, 96, 0.232, 1, 0)
(11, 30, 4, 37.6, 0, 110, 92, 0.191, 0, 0)
(12, 34, 10, 38.0, 0, 168, 74, 0.537, 1, 0)
(13, 57, 10, 27.1, 0, 139, 80, 1.441, 0, 0)
(14, 59, 1, 30.1, 846, 189, 60, 0.398, 1, 23)
(15, 51, 5, 25.8, 175, 166, 72, 0.587, 1, 19)
(16, 32, 7, 30.0, 0, 100, 0, 0.484, 1, 0)
(17, 31, 0, 45.8, 230, 118, 84, 0.551, 1, 47)
(18, 31, 7, 29.6, 0, 107, 74, 0.254, 1, 0)
(19, 33, 1, 43.3, 83, 103, 30, 0.183, 0, 38)
(20, 32, 1, 34.6, 96, 115, 70, 0.529, 1, 30)
(21, 27, 3, 39.3, 235, 126, 88, 0.704, 0, 41)
(22, 50, 8, 35.4, 0, 99, 84, 0.388, 0, 0)
(23, 41, 7, 39.8, 0

In [ ]:
# --- Always close the connection when done ---
conn.close()
print("Connection closed.")

In [14]:
# --- Import SQLAlchemy ---
from sqlalchemy import create_engine

# --- Load the connection string from .env ---
load_dotenv()
DB_STRING = os.getenv("DB_STRING")

# --- Create an engine from the connection string ---
# The engine is a factory that manages database connections
db = create_engine(DB_STRING)
print("Engine created!")

Engine created!


In [15]:
# --- Load the full kaggle_survey table into a DataFrame ---
query = """select p.*, bm.insulin, bm.glucose , bm.bloodpressure,
po.diabetespedigreefunction , po.outcome,
sk.skinthickness 
from diabetes.patient p
inner join diabetes.blood_metrics bm on bm.patientid = p.id
inner join diabetes.pedigree_outcome po on po.patientid = p.id
inner join diabetes.skin sk on sk.patientid = p.id; 
"""
df_sqlalchemy = pd.read_sql(query, db)

print(f"Shape: {df_sqlalchemy.shape}")
df_sqlalchemy.head()

Shape: (1536, 10)


,id,Age,pregnancies,bmi,insulin,glucose,bloodpressure,diabetespedigreefunction,outcome,skinthickness
0,1,50,6,33.6,0,148,72,0.627,1,35
1,2,31,1,26.6,0,85,66,0.351,0,29
2,3,32,8,23.3,0,183,64,0.672,1,0
3,4,21,1,28.1,94,89,66,0.167,0,23
4,5,33,0,43.1,168,137,40,2.288,1,35


In [20]:
# --- Export to CSV ---
# index=False prevents pandas from writing the row numbers as a column
df_sqlalchemy.to_csv("data/diabetes_data.csv", index=False)

print("Saved to data/diabetes_data.csv")
print(
    f"File contains {len(df_sqlalchemy):,} rows and {len(df_sqlalchemy.columns)} columns"
)

Saved to data/diabetes_data.csv
File contains 1,536 rows and 10 columns


In [ ]:
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit

# Load the data
df = pd.read_csv("data/diabetes_data.csv")

# Separate features and target (drop the ID so the model can't learn from it)
X = df.drop(columns=["outcome", "id"])
y = df["outcome"]
groups = df["id"]

# Split so that all rows of a patient end up in the same set
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

# Save the splits (full rows, including patientid and outcome)
df.iloc[train_idx].to_csv("data/train.csv", index=False)
df.iloc[test_idx].to_csv("data/test.csv", index=False)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print("Patient overlap:", set(groups.iloc[train_idx]) & set(groups.iloc[test_idx]))

Train: (1228, 8), Test: (308, 8)
Patient overlap: set()
